# Arabic Text Classification — Notebook 1: Preprocessing

**Purpose:** Load the raw dataset, apply all text cleaning steps, and save a clean CSV file.


**Output:** `outputs/cleaned_dataset.csv` with columns: `clean_text`, `label`

---
## 1. Imports & Setup

In [ ]:
import os
import re
import json
import warnings
import unicodedata

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)

# Optional: emoji library for better emoji handling
try:
    import emoji as emoji_lib
    EMOJI_OK = True
except ImportError:
    EMOJI_OK = False
    print("[WARN] `emoji` not installed — run: pip install emoji")

# Optional: ftfy for fixing encoding corruption (mojibake)
try:
    import ftfy
    FTFY_OK = True
except ImportError:
    FTFY_OK = False
    print("[WARN] `ftfy` not installed — run: pip install ftfy  (strongly recommended)")

print("[OK] Imports complete.")

[WARN] `emoji` not installed — run: pip install emoji
[WARN] `ftfy` not installed — run: pip install ftfy  (strongly recommended)
[OK] Imports complete.


---
## 2. Configuration

Adjust these variables to match your dataset.

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
Dataset_path = "/content/Multi-dialect Arabic dataset.csv"   # path to raw file (.xlsx or .csv)
SHEET_INDEX    = 17                                  # Excel sheet index (ignored for CSV)
TEXT_COL       = "Comment"                           # column containing raw Arabic text
LABEL_COL      = "label"                             # column containing class labels
DIALECT_COL    = "dialect"                           # column containing dialect labels (set to None to skip)

# ── Cleaning options ─────────────────────────────────────────────────────────
EMOJI_MODE     = "remove"    # "remove" | "demojize"  (demojize converts 😊 → :smiling_face:)
DROP_NEUTRAL   = True        # whether to discard rows labelled "neutral"
DROP_DUPES     = True        # whether to drop duplicate texts

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_CSV     = "outputs/cleaned_dataset.csv"

---
## 3. Compiled Regex Patterns

Pre-compiled once at module level for efficiency.

In [ ]:
# Arabic-Indic digit translation tables
_AR_INDIC     = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
_EXT_AR_INDIC = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")

RE_URL       = re.compile(r"https?://\S+|www\.\S+")
RE_MENTION   = re.compile(r"@\w+")
RE_HASHTAG   = re.compile(r"#(\w+)")
RE_FLAG      = re.compile(r"[\U0001F1E6-\U0001F1FF]{2}")        # flag emoji pairs
RE_EMOJI_FB  = re.compile(r"[\U00010000-\U0010ffff]", flags=re.UNICODE)  # fallback emoji
RE_TASHKEEL  = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]")  # diacritics
RE_TATWEEL   = re.compile(r"\u0640")                             # Arabic tatweel (kashida)
RE_OBFUSC_AR = re.compile(r"(?<=[\u0600-\u06FF])([^\u0600-\u06FF\w\s]+)(?=[\u0600-\u06FF])")  # obfuscation
RE_ELONGATE  = re.compile(r"(.)\1{3,}")                         # repeated chars (ههههه)
RE_REPUNCT   = re.compile(r"([!?.,،؛؟]){2,}")                   # repeated punctuation

print("[OK] Regex patterns compiled.")

[OK] Regex patterns compiled.


---
## 4. Encoding Fix Utilities

Handles the common case where Arabic text stored in Excel/CSV files becomes garbled  
(mojibake: e.g., `ماشاء` appears as `Ù…Ø§Ø´Ø§Ø¡`).  
This must run **before** any cleaning step.

In [ ]:
def fix_encoding(text: str) -> str:
    """
    Fix mojibake caused by reading UTF-8 encoded Excel/CSV as Latin-1.
    Tries ftfy first, then manual re-encoding strategies.
    """
    if not isinstance(text, str) or text.strip() == "":
        return ""

    # Strategy 1: ftfy (best library for this exact problem)
    if FTFY_OK:
        fixed = ftfy.fix_text(text)
        if fixed != text and re.search(r"[\u0600-\u06FF]", fixed):
            return fixed

    # Strategy 2: Manual re-encode (latin-1 bytes → utf-8 string)
    try:
        fixed = text.encode("latin-1").decode("utf-8")
        if re.search(r"[\u0600-\u06FF]", fixed):
            return fixed
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    # Strategy 3: Try cp1252 (Windows Western European — common in Excel files)
    try:
        fixed = text.encode("cp1252").decode("utf-8")
        if re.search(r"[\u0600-\u06FF]", fixed):
            return fixed
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    return text  # already fine or unfixable


def detect_and_fix_encoding(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    """
    Check whether the text column has mojibake and fix it if needed.
    Prints a clear diagnostic so you can see exactly what happened.
    """
    sample = df[text_col].dropna().iloc[0] if len(df) > 0 else ""
    has_arabic   = bool(re.search(r"[\u0600-\u06FF]", str(sample)))
    has_mojibake = bool(re.search(r"Ø|Ù|Ú|ð|â", str(sample)))

    if has_mojibake and not has_arabic:
        print(f"[ENCODING] WARNING: Mojibake detected in '{text_col}' column!")
        print(f"           Sample (broken): {str(sample)[:80]}")
        df = df.copy()
        df[text_col] = df[text_col].fillna("").astype(str).apply(fix_encoding)
        fixed_sample = df[text_col].iloc[0]
        print(f"           Sample (fixed) : {str(fixed_sample)[:80]}")
        arabic_ok = df[text_col].apply(lambda x: bool(re.search(r"[\u0600-\u06FF]", x))).mean()
        print(f"           Arabic found in {arabic_ok*100:.1f}% of rows after fix.")
    elif has_arabic:
        print(f"[ENCODING] OK — Arabic text detected correctly.")
    else:
        print(f"[ENCODING] WARNING — Could not auto-detect. Sample: {str(sample)[:80]}")

    return df

---
## 5. Text Cleaning Functions

All cleaning steps are separated into small, single-purpose functions  
and composed in `clean_arabic_text()` at the bottom.

In [ ]:
def unicode_normalize(text: str) -> str:
    """NFKC normalization + collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", text).strip()


def convert_arabic_indic_digits(text: str) -> str:
    """Convert Arabic-Indic (٠١٢…) and Extended Arabic-Indic (۰۱۲…) digits to ASCII."""
    return text.translate(_AR_INDIC).translate(_EXT_AR_INDIC)


def remove_flag_emojis(text: str) -> str:
    """Remove flag emoji pairs (e.g., 🇸🇦) from text."""
    return RE_FLAG.sub(" ", text)


def handle_emojis(text: str, mode: str = "remove") -> str:
    """
    Handle non-flag emojis.
    mode='remove'   → strip all emojis (uses emoji lib if available, else regex fallback)
    mode='demojize' → convert to text descriptions (e.g., 😊 → :smiling_face:)
    """
    if mode == "demojize" and EMOJI_OK:
        return emoji_lib.demojize(text, language="en")
    if EMOJI_OK:
        return emoji_lib.replace_emoji(text, replace=" ")
    return RE_EMOJI_FB.sub(" ", text)  # regex fallback


def remove_urls_and_mentions(text: str) -> str:
    """Remove HTTP/HTTPS URLs and @mentions."""
    text = RE_URL.sub(" ", text)
    text = RE_MENTION.sub(" ", text)
    return text


def expand_hashtags(text: str) -> str:
    """Strip the # sign but keep the hashtag word (e.g., #السعودية → السعودية)."""
    return RE_HASHTAG.sub(r" \1 ", text)


def remove_tashkeel(text: str) -> str:
    """Remove Arabic diacritics (tashkeel) and tatweel (kashida stretching character)."""
    text = RE_TASHKEEL.sub("", text)
    text = RE_TATWEEL.sub("", text)
    return text


def normalize_arabic_letters(text: str) -> str:
    """
    Light normalization of Arabic letter variants:
    - Alef variants (إ أ ٱ آ) → ا
    - Alef maqsura (ى) → ي
    """
    text = re.sub(r"[إأٱآ]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    return text


def deobfuscate_arabic(text: str) -> str:
    """
    Remove punctuation/symbols injected between Arabic characters
    to evade hate-speech filters (e.g., ك*ل*ب → كلب).
    Applied iteratively until stable.
    """
    prev = None
    while prev != text:
        prev = text
        text = RE_OBFUSC_AR.sub("", text)
    return text


def fix_fragmented_arabic(text: str) -> str:
    """
    Re-join runs of single Arabic characters that were split by spaces
    (common artifact of some scrapers).
    e.g., 'م ر ح ب ا' → 'مرحبا'
    """
    tokens = text.split(" ")
    result = []
    i = 0
    while i < len(tokens):
        tok = tokens[i]
        if len(tok) == 1 and re.match(r'[\u0600-\u06FF]', tok):
            run = [tok]
            j = i + 1
            while j < len(tokens) and len(tokens[j]) == 1 and re.match(r'[\u0600-\u06FF]', tokens[j]):
                run.append(tokens[j])
                j += 1
            result.append("".join(run) if len(run) >= 2 else tok)
            i = j if len(run) >= 2 else i + 1
        else:
            result.append(tok)
            i += 1
    return " ".join(result)


def collapse_elongation(text: str) -> str:
    """Collapse 4+ repeated characters to 3 (e.g., هههههه → ههه)."""
    return RE_ELONGATE.sub(r"\1\1\1", text)


def clean_repeated_punctuation(text: str) -> str:
    """Collapse repeated punctuation (e.g., !!! → !, ؟؟؟ → ؟)."""
    return RE_REPUNCT.sub(r"\1", text)


# ── Master cleaning pipeline ──────────────────────────────────────────────────
def clean_arabic_text(text: str, emoji_mode: str = "remove") -> str:
    """
    Full Arabic text cleaning pipeline.
    Returns a cleaned string. Does NOT tokenize.

    Steps (in order):
    1. Unicode normalization (NFKC)
    2. Arabic-Indic digit conversion
    3. Flag emoji removal
    4. Emoji handling (remove or demojize)
    5. URL and mention removal
    6. Hashtag expansion
    7. Deobfuscation (remove symbols injected between Arabic letters)
    8. Fragment rejoining (re-join spaced single Arabic chars)
    9. Tashkeel and tatweel removal
    10. Light letter normalization (alef variants, alef maqsura)
    11. Elongation collapsing
    12. Repeated punctuation collapsing
    13. Final whitespace cleanup
    """
    text = unicode_normalize(text)
    text = convert_arabic_indic_digits(text)
    text = remove_flag_emojis(text)
    text = handle_emojis(text, mode=emoji_mode)
    text = remove_urls_and_mentions(text)
    text = expand_hashtags(text)
    text = deobfuscate_arabic(text)
    text = fix_fragmented_arabic(text)
    text = remove_tashkeel(text)
    text = normalize_arabic_letters(text)
    text = collapse_elongation(text)
    text = clean_repeated_punctuation(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("[OK] Cleaning functions defined.")

[OK] Cleaning functions defined.


---
## 6. Load Raw Dataset

In [ ]:
print(f"[INFO] Loading: {Dataset_path}")

if Dataset_path.endswith(".xlsx"):
    df_raw = pd.read_excel(Dataset_path, sheet_name=SHEET_INDEX)
else:
    df_raw = pd.read_csv(Dataset_path, encoding="utf-8-sig")

print(f"[INFO] Shape     : {df_raw.shape}")
print(f"[INFO] Columns   : {df_raw.columns.tolist()}")
print(f"[INFO] Missing values:\n{df_raw.isnull().sum()}\n")

print(f"[INFO] Label distribution:")
print(df_raw[LABEL_COL].value_counts())
print()
df_raw.head(3)

[INFO] Loading: /content/Multi-dialect Arabic dataset.csv
[INFO] Shape     : (7624, 5)
[INFO] Columns   : ['Comment', 'dialect', 'platform', 'Category', 'label']
[INFO] Missing values:
Comment     0
dialect     0
platform    7
Category    0
label       0
dtype: int64

[INFO] Label distribution:
label
P          3629
N          3625
Neutral     370
Name: count, dtype: int64



,Comment,dialect,platform,Category,label
0,خطية صغيرة بعدها وهذا اخوها على اساس غسل عار,Iraq,YT,Feminist,P
1,شحلاتها,Iraq,YT,Feminist,P
2,احبج,Iraq,YT,Feminist,P


---
## 7. Exploratory Data Analysis (EDA)

Quick visual summary of the raw dataset before cleaning.

In [ ]:
def run_eda(df: pd.DataFrame, text_col: str, label_col: str, save_path: str):
    """Plot label distribution and text-length histogram."""
    has_extra = "platform" in df.columns or "dialect" in df.columns
    n_plots = 3 if has_extra else 2
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
    fig.suptitle("EDA — Arabic Sentiment Dataset (Raw)", fontsize=14, fontweight="bold")

    # Label distribution
    label_counts = df[label_col].value_counts()
    axes[0].pie(label_counts.values, labels=label_counts.index, autopct="%1.1f%%",
                colors=["#4CAF50", "#F44336", "#2196F3"], startangle=140)
    axes[0].set_title("Label Distribution")

    # Text length distribution
    word_counts = df[text_col].fillna("").apply(lambda x: len(x.split()))
    for lbl in df[label_col].unique():
        sub = word_counts[df[label_col] == lbl]
        axes[1].hist(sub, bins=40, alpha=0.6, label=str(lbl))
    axes[1].set_title("Text Length by Label (words)")
    axes[1].set_xlabel("Word count")
    axes[1].axvline(128, color="red",    linestyle="--", alpha=0.8, label="128")
    axes[1].axvline(256, color="orange", linestyle="--", alpha=0.8, label="256")
    axes[1].legend(fontsize=8)

    # Optional: platform or dialect breakdown
    if has_extra and n_plots == 3:
        col = "platform" if "platform" in df.columns else "dialect"
        counts = df[col].value_counts().head(10)
        axes[2].barh(counts.index.astype(str), counts.values, color="#5C85D6")
        axes[2].invert_yaxis()
        axes[2].set_title(f"By {col.capitalize()}")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[SAVED] EDA chart → {save_path}")


run_eda(df_raw, text_col=TEXT_COL, label_col=LABEL_COL, save_path="outputs/eda_raw.png")

[SAVED] EDA chart → outputs/eda_raw.png


---
## 8. Preprocessing Pipeline

Steps applied in order:
1. Fix encoding (mojibake)
2. Drop duplicates (optional)
3. Drop neutral-labelled rows (optional)
4. Clean text → `clean_text` column
5. Drop rows that are empty after cleaning

In [ ]:
# Include dialect column if configured
_cols = [TEXT_COL, LABEL_COL]
if DIALECT_COL and DIALECT_COL in df_raw.columns:
    _cols.append(DIALECT_COL)
elif DIALECT_COL:
    print(f"[WARN] DIALECT_COL='{DIALECT_COL}' not found in dataset. Available: {df_raw.columns.tolist()}")
    DIALECT_COL = None

df = df_raw[_cols].copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

print(f"[STEP 0] Initial shape: {df.shape}")
print(f"         Labels:\n{df[LABEL_COL].value_counts()}\n")


GULF_STATES = ["Emirates", "Qaṭar", "Oman", "Bahrain", "Kuwait"]

# ── combine gulf ──────────────────────────────────────────────────────
df["dialect"] = df["dialect"].replace(
    {state: "Gulf" for state in GULF_STATES}
)

# ── Step 1: Fix encoding ──────────────────────────────────────────────────────
df = detect_and_fix_encoding(df, TEXT_COL)

# ── Step 2: Drop duplicates ───────────────────────────────────────────────────
if DROP_DUPES:
    before = len(df)
    df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
    print(f"[STEP 2] Dropped {before - len(df)} duplicate rows. Remaining: {len(df)}")

# ── Step 3: Drop neutral rows ─────────────────────────────────────────────────
if DROP_NEUTRAL:
    neutral_mask = df[LABEL_COL].str.strip().str.lower().isin(["neutral"])
    n_neutral = neutral_mask.sum()
    df = df[~neutral_mask].reset_index(drop=True)
    print(f"[STEP 3] Dropped {n_neutral} neutral rows. Remaining: {len(df)}")

# ── Step 4: Remove rows that are already empty pre-cleaning ───────────────────
df = df[df[TEXT_COL].str.strip() != ""].reset_index(drop=True)

# ── Step 5: Apply text cleaning ───────────────────────────────────────────────
print("[STEP 5] Cleaning text (this may take a moment) ...")
df["clean_text"] = df[TEXT_COL].apply(lambda t: clean_arabic_text(t, emoji_mode=EMOJI_MODE))

# ── Step 6: Drop rows that became empty after cleaning ────────────────────────
empty_mask = df["clean_text"].str.strip() == ""
print(f"[STEP 6] Removed {empty_mask.sum()} rows that became empty after cleaning.")
df = df[~empty_mask].reset_index(drop=True)

print(f"\n[INFO] Final shape: {df.shape}")
print(f"[INFO] Label distribution:\n{df[LABEL_COL].value_counts()}\n")

[STEP 0] Initial shape: (7624, 3)
         Labels:
label
P          3629
N          3625
Neutral     370
Name: count, dtype: int64

[ENCODING] OK — Arabic text detected correctly.
[STEP 2] Dropped 15 duplicate rows. Remaining: 7609
[STEP 3] Dropped 369 neutral rows. Remaining: 7240
[STEP 5] Cleaning text (this may take a moment) ...
[STEP 6] Removed 0 rows that became empty after cleaning.

[INFO] Final shape: (7240, 4)
[INFO] Label distribution:
label
P    3624
N    3616
Name: count, dtype: int64



---
## 9. Quality Check

Verify a sample of cleaned texts look correct before saving.

In [ ]:
print("[VERIFY] Random sample of cleaned texts:")
sample = df[[TEXT_COL, "clean_text", LABEL_COL]].sample(min(5, len(df)), random_state=42)
for _, row in sample.iterrows():
    print(f"  RAW  : {str(row[TEXT_COL])[:100]}")
    print(f"  CLEAN: {str(row['clean_text'])[:100]}")
    print(f"  LABEL: {row[LABEL_COL]}")
    print()

# Basic stats on cleaned text length
lengths = df["clean_text"].apply(lambda x: len(x.split()))
print(f"[STATS] Clean text word counts:")
print(f"        mean={lengths.mean():.1f} | median={lengths.median():.0f} | "
      f"max={lengths.max()} | min={lengths.min()}")
print(f"        % with <5 words : {(lengths < 5).mean()*100:.1f}%")
print(f"        % with >256 words: {(lengths > 256).mean()*100:.1f}%")

[VERIFY] Random sample of cleaned texts:
  RAW  : موحجاها القائد كاللهم تنذلون
  CLEAN: موحجاها القائد كاللهم تنذلون
  LABEL: N

  RAW  : هذي قمامة اليمن .
  CLEAN: هذي قمامة اليمن .
  LABEL: N

  RAW  : الصراحه مكنعرفكش ولكن شرفتي بالمرأة المغربية المسلمة ،الله يوفقك أختي 👏👏👏😍
  CLEAN: الصراحه مكنعرفكش ولكن شرفتي بالمراة المغربية المسلمة ،الله يوفقك اختي
  LABEL: P

  RAW  : مصنع قذارة وزباله
  CLEAN: مصنع قذارة وزباله
  LABEL: N

  RAW  : شكون.هي.لي.متحضرش.واش.عندها.حصانة.هادي.غي.ميكروب.وتستحمر.المغفلين.
  CLEAN: شكونهيليمتحضرشواشعندهاحصانةهاديغيميكروبوتستحمرالمغفلين.
  LABEL: N

[STATS] Clean text word counts:
        mean=14.1 | median=10 | max=316 | min=1
        % with <5 words : 15.2%
        % with >256 words: 0.0%


---
## 10. Shuffle

Shuffle the cleaned dataset with a fixed random seed before splitting.  
This removes any ordering bias that may exist in the raw data (e.g., all positives first).

In [ ]:
RANDOM_SEED = 42
_keep_cols = ["clean_text", LABEL_COL]
if DIALECT_COL:
    _keep_cols.append(DIALECT_COL)

df_clean = df[_keep_cols].rename(columns={LABEL_COL: "label"}).copy()

df_clean = df_clean.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"[SHUFFLE] Dataset shuffled with random_state={RANDOM_SEED}.")
print(f"          Shape: {df_clean.shape}")
print(f"          Label distribution after shuffle:\n{df_clean['label'].value_counts()}\n")

[SHUFFLE] Dataset shuffled with random_state=42.
          Shape: (7240, 3)
          Label distribution after shuffle:
label
P    3624
N    3616
Name: count, dtype: int64



---
## 11. Label Encoding

Convert string labels (e.g., `positive`, `negative`) to integer IDs.  
Both the original string label and the integer `label_id` are kept in the dataset.  
The mapping is saved to `outputs/label_map.json` for use during training and inference.

If a `dialect` column is present, it is also extracted and encoded with a separate `LabelEncoder`.  
The dialect mapping is saved to `outputs/dialect_map.json`.


In [ ]:
# ── Sentiment label encoding ─────────────────────────────────────────────────
le = LabelEncoder()
df_clean["labels"] = le.fit_transform(df_clean["label"])

class_names = le.classes_.tolist()           # e.g., ['negative', 'positive']
label2id    = {c: int(i) for i, c in enumerate(class_names)}
id2label    = {int(i): c for i, c in enumerate(class_names)}

print(f"[LABEL] Classes  : {class_names}")
print(f"[LABEL] label2id : {label2id}")
print(f"[LABEL] id2label : {id2label}")
print()
print(df_clean[["label", "labels"]].value_counts().sort_index().to_string())

# Save label map
os.makedirs("outputs", exist_ok=True)
with open("outputs/label_map.json", "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, ensure_ascii=False, indent=2)
print(f"[SAVED] Label map → outputs/label_map.json")

# Drop original string label column
df_clean = df_clean.drop(columns="label")

# ── Dialect encoding ──────────────────────────────────────────────────────────
if DIALECT_COL and DIALECT_COL in df_clean.columns:
    print()
    # Drop rows with missing dialect values
    missing_dialect = df_clean[DIALECT_COL].isna() | (df_clean[DIALECT_COL].astype(str).str.strip() == "")
    if missing_dialect.sum() > 0:
        print(f"[DIALECT] Dropping {missing_dialect.sum()} rows with missing dialect values.")
        df_clean = df_clean[~missing_dialect].reset_index(drop=True)

    le_dialect = LabelEncoder()
    df_clean["dialect_id"] = le_dialect.fit_transform(df_clean[DIALECT_COL].astype(str))

    dialect_names  = le_dialect.classes_.tolist()
    dialect2id     = {c: int(i) for i, c in enumerate(dialect_names)}
    id2dialect     = {int(i): c for i, c in enumerate(dialect_names)}

    print(f"[DIALECT] Classes  : {dialect_names}")
    print(f"[DIALECT] dialect2id: {dialect2id}")
    print()
    print(df_clean[[DIALECT_COL, "dialect_id"]].value_counts().sort_index().to_string())

    # Save dialect map
    with open("outputs/dialect_map.json", "w", encoding="utf-8") as f:
        json.dump({"dialect2id": dialect2id, "id2dialect": id2dialect}, f, ensure_ascii=False, indent=2)
    print(f"[SAVED] Dialect map → outputs/dialect_map.json")

    # Drop original string dialect column
    df_clean = df_clean.drop(columns=DIALECT_COL)
else:
    print("[DIALECT] Skipped — DIALECT_COL not set or not found in dataframe.")

[LABEL] Classes  : ['N', 'P']
[LABEL] label2id : {'N': 0, 'P': 1}
[LABEL] id2label : {0: 'N', 1: 'P'}

label  labels
N      0         3616
P      1         3624
[SAVED] Label map → outputs/label_map.json

[DIALECT] Classes  : ['Algeria', 'Egypt', 'Gulf', 'Iraq', 'Jordan', 'Lebanon', 'Libya', 'Moroccan', 'Palestine', 'Saudi', 'Sudanese', 'Syria', 'Tunisia', 'Yemen']
[DIALECT] dialect2id: {'Algeria': 0, 'Egypt': 1, 'Gulf': 2, 'Iraq': 3, 'Jordan': 4, 'Lebanon': 5, 'Libya': 6, 'Moroccan': 7, 'Palestine': 8, 'Saudi': 9, 'Sudanese': 10, 'Syria': 11, 'Tunisia': 12, 'Yemen': 13}

dialect    dialect_id
Algeria    0             527
Egypt      1             527
Gulf       2             389
Iraq       3             527
Jordan     4             528
Lebanon    5             525
Libya      6             528
Moroccan   7             527
Palestine  8             526
Saudi      9             528
Sudanese   10            529
Syria      11            527
Tunisia    12            524
Yemen      13         

---
## 12. Train / Validation / Test Split (70 / 15 / 15)

Stratified split to preserve the class distribution across all three sets.  
Each split is saved as its own CSV file so the training notebook can load them directly.

In [ ]:
TEST_SIZE = 0.15
VAL_SIZE = 0.15

# ── Step 1: carve out test set (15%) ─────────────────────────────────────────
df_trainval, df_test = train_test_split(
    df_clean,
    test_size=TEST_SIZE,
    stratify=df_clean["dialect_id"],
    random_state=RANDOM_SEED,
)

# ── Step 2: split remainder into train (70%) and val (15%) ───────────────────
# val_size relative to the remaining 85% so that it equals 15% of the full set
relative_val_size = VAL_SIZE / (1.0 - TEST_SIZE)

df_train, df_val = train_test_split(
    df_trainval,
    test_size=relative_val_size,
    stratify=df_trainval["dialect_id"],
    random_state=RANDOM_SEED,
)

# Reset indices for clean CSVs
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

total = len(df_clean)
print(f"[SPLIT] Total   : {total}")
print(f"        Train   : {len(df_train):>5}  ({len(df_train)/total*100:.1f}%)")
print(f"        Val     : {len(df_val):>5}  ({len(df_val)/total*100:.1f}%)")
print(f"        Test    : {len(df_test):>5}  ({len(df_test)/total*100:.1f}%)")
print()

# Verify class balance is preserved in each split
for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    dist = split["labels"].value_counts(normalize=True).mul(100).round(1).to_dict()
    print(f"  {name} labels %: {dist}")

[SPLIT] Total   : 7240
        Train   :  5068  (70.0%)
        Val     :  1086  (15.0%)
        Test    :  1086  (15.0%)

  Train labels %: {1: 50.8, 0: 49.2}
  Val labels %: {0: 51.0, 1: 49.0}
  Test labels %: {0: 52.5, 1: 47.5}


---
## 13. Save All Outputs

Saves the full cleaned dataset plus the three split files.  
The training notebook only needs to load `train.csv`, `val.csv`, and `test.csv`.

In [ ]:
OUTPUT_DIR ="/content/outputs"


from google.colab import files

# Full cleaned dataset (useful for inspection or reuse)
df_clean.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"[SAVED] Full cleaned dataset → {OUTPUT_CSV}  ({df_clean.shape})")

# Individual split files
train_path = f"{OUTPUT_DIR}/train.csv"
val_path   = f"{OUTPUT_DIR}/val.csv"
test_path  = f"{OUTPUT_DIR}/test.csv"

df_train.to_csv(train_path, index=False, encoding="utf-8-sig")
df_val.to_csv(val_path,   index=False, encoding="utf-8-sig")
df_test.to_csv(test_path,  index=False, encoding="utf-8-sig")

print(f"[SAVED] Train split → {train_path}  ({df_train.shape})")
print(f"[SAVED] Val split   → {val_path}   ({df_val.shape})")
print(f"[SAVED] Test split  → {test_path}  ({df_test.shape})")
print()
print("[DONE] Preprocessing complete.")
print("       Columns in each file:", df_train.columns.tolist())
if "dialect_id" in df_train.columns:
    print(f"       Dialect classes   : {list(id2dialect.values()) if 'id2dialect' in dir() else 'see outputs/dialect_map.json'}")
print("       Next step: open 02_training.ipynb and load train/val/test.csv from outputs/")

[SAVED] Full cleaned dataset → outputs/cleaned_dataset.csv  ((7240, 3))
[SAVED] Train split → /content/outputs/train.csv  ((5068, 3))
[SAVED] Val split   → /content/outputs/val.csv   ((1086, 3))
[SAVED] Test split  → /content/outputs/test.csv  ((1086, 3))

[DONE] Preprocessing complete.
       Columns in each file: ['clean_text', 'labels', 'dialect_id']
       Dialect classes   : ['Algeria', 'Egypt', 'Gulf', 'Iraq', 'Jordan', 'Lebanon', 'Libya', 'Moroccan', 'Palestine', 'Saudi', 'Sudanese', 'Syria', 'Tunisia', 'Yemen']
       Next step: open 02_training.ipynb and load train/val/test.csv from outputs/
